# VisionGuard — Colab GPU (fast + evidence-first)

**Runtime → Change runtime type → GPU** (T4 is enough for short clips).

| Goal | How |
|------|-----|
| Fast | YOLO nano + GPU SigLIP + `WIN_SEC=8` + parallel NVIDIA workers |
| Grounded | Object answers from detector facts only |
| Anti-hallucination check | Absent object must return `insufficient_evidence=true` |

**Honest limit:** no ML stack is zero-error. This project refuses silent invention as verified fact.

Set Colab **Secrets** → `NVIDIA_API_KEY` first.

## 1) GPU check

In [ ]:
import sys, torch
print(sys.version)
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM_GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

## 2) Repository root

Clone/upload the project so `run.py` is in the working directory.

In [ ]:
# Option A — GitHub:
# %cd /content
# !git clone https://github.com/YOUR_USER/visionguard-ai.git
# %cd /content/visionguard-ai

# Option B — uploaded zip:
# %cd /content
# !unzip -q visionguard-ai.zip
# %cd /content/visionguard-ai

import os
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "run.py").exists() and (ROOT / "visionguard-ai" / "run.py").exists():
    os.chdir(ROOT / "visionguard-ai")
    ROOT = Path.cwd()
assert (ROOT / "run.py").exists(), f"cd into the repo root first (cwd={ROOT})"
print("ROOT:", ROOT)

## 3) Install

In [ ]:
!apt-get -qq update && apt-get -qq install -y libgl1 libglib2.0-0 > /dev/null
!pip -q install -r requirements.txt
print("deps ok")

## 4) Secrets + fast GPU profile + models

In [ ]:
import os, sys, subprocess
from pathlib import Path

try:
    from google.colab import userdata
    key = userdata.get("NVIDIA_API_KEY")
except Exception:
    key = os.environ.get("NVIDIA_API_KEY", "")

assert key and str(key).startswith("nvapi-"), "Set Colab Secret NVIDIA_API_KEY (nvapi-...)"
os.environ["NVIDIA_API_KEY"] = key

ROOT = Path.cwd()
subprocess.check_call([
    sys.executable, "scripts/apply_colab_fast_env.py",
    "--from-env", "--write-dotenv", "--device", "cuda",
], cwd=str(ROOT))
subprocess.check_call([
    sys.executable, "scripts/bootstrap_models.py", "--yolo", "yolo11n.pt",
], cwd=str(ROOT))

from visionguard.runtime.env import load_project_env
load_project_env(ROOT)

print("Preloading SigLIP (first time downloads weights)...")
from transformers import AutoModel, AutoProcessor
name = os.environ.get("CLIP_MODEL", "google/siglip2-so400m-patch14-384")
AutoProcessor.from_pretrained(name)
AutoModel.from_pretrained(name)
print("Models ready | device=", os.environ.get("VISION_GUARD_DEVICE"),
      "| YOLO=", os.environ.get("YOLO_MODEL"),
      "| WIN_SEC=", os.environ.get("WIN_SEC"),
      "| SEMANTIC_WORKERS=", os.environ.get("SEMANTIC_WORKERS"))

## 5) Index once + anti-hallucination checks (in-process)

Keeps the Flask app and GPU models warm for interactive queries in the next cell.
Use `sample_videos/asset3.mp4` for the fastest demo.

In [ ]:
import io, json, os, time
from pathlib import Path
from visionguard.runtime.env import load_project_env
from visionguard.web_app.server import create_app

load_project_env(Path.cwd())
os.environ["VISION_GUARD_SKIP_WARMUP"] = "1"
os.environ["VERIFIER_READY_TIMEOUT"] = "0"

VIDEO = Path("sample_videos/asset3.mp4")
PRESENT_Q = "find the person"
ABSENT_Q = "find the elephant"
assert VIDEO.is_file(), VIDEO

app = create_app(testing=True, start_warmup=False)
client = app.test_client()

t0 = time.perf_counter()
up = client.post(
    "/api/videos/upload",
    data={"video": (io.BytesIO(VIDEO.read_bytes()), VIDEO.name)},
    content_type="multipart/form-data",
)
assert up.status_code == 201, up.get_data(as_text=True)
ids = up.get_json()
video_id = ids["video_id"]
print("video_id:", video_id)

ix = client.post(f"/api/videos/{video_id}/index")
assert ix.status_code == 202, ix.get_data(as_text=True)

while True:
    st = client.get(f"/api/videos/{video_id}/status").get_json()
    stages = ", ".join(f"{s['name']}={s['status']}" for s in st["stages"])
    print(f"status={st['status']} | {stages}")
    if st["status"] in {"completed", "failed"}:
        break
    time.sleep(1)
assert st["status"] == "completed", st

found = client.post(f"/api/videos/{video_id}/query", json={"query": PRESENT_Q}).get_json()
absent = client.post(f"/api/videos/{video_id}/query", json={"query": ABSENT_Q}).get_json()
elapsed = round(time.perf_counter() - t0, 2)

checks = {
    "present_has_evidence": bool(found.get("frames")) and found.get("insufficient_evidence") is False,
    "absent_abstains": absent.get("frames") == [] and absent.get("insufficient_evidence") is True,
    "semantic_completed": next(s for s in st["stages"] if s["name"] == "semantic_analysis")["status"] == "completed",
}
report = {
    "video": str(VIDEO),
    "video_id": video_id,
    "elapsed_sec": elapsed,
    "checks": checks,
    "all_passed": all(checks.values()),
    "present_answer": (found.get("answer") or found.get("message") or "")[:300],
    "absent_answer": (absent.get("answer") or absent.get("message") or "")[:300],
}
Path("output").mkdir(exist_ok=True)
Path("output/colab_e2e_report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
assert report["all_passed"], "Grounding checks failed — do not trust this run"
print(f"\nPASS in {elapsed}s — client/video_id kept for next cell")

## 6) Interactive grounded queries

Reuses `client` + `video_id` from cell 5 (do not restart the runtime).

**Prefer:** `find the person`, `find the car`, `how many people`  
**Avoid for strict grounding:** speech, identity, “suspicious”, free-form scene poetry

In [ ]:
QUERIES = [
    "find the person",
    "how many people",
    "find the car",
    "find the elephant",  # must abstain
]

for q in QUERIES:
    payload = client.post(f"/api/videos/{video_id}/query", json={"query": q}).get_json()
    print("\n===" , q, "===")
    print("insufficient_evidence:", payload.get("insufficient_evidence"))
    print("frames:", len(payload.get("frames") or []))
    print("answer:", (payload.get("answer") or payload.get("message") or "")[:240])
    for m in (payload.get("matches") or [])[:2]:
        print(
            "  t",
            m.get("start"), "→", m.get("end"),
            "objects=", m.get("objects"),
            "provenance=", m.get("claim_provenance") or m.get("evidence_state"),
        )

## 7) Optional one-liner smoke test (subprocess)

Useful if you only want a scripted PASS/FAIL without interactive cells:

```bash
python scripts/run_colab_e2e.py --video sample_videos/asset3.mp4
```

## Optional UI

Uncomment the next cell and add Secret `NGROK_AUTHTOKEN` if you need the browser UI.

In [ ]:
# import os, threading
# from pathlib import Path
# from visionguard.runtime.env import load_project_env
# load_project_env(Path.cwd())
# os.environ["VISION_GUARD_HOST"] = "0.0.0.0"
# from visionguard.web_app.server import app as ui_app
# threading.Thread(
#     target=lambda: ui_app.run(host="0.0.0.0", port=7860, debug=False, threaded=True, load_dotenv=False),
#     daemon=True,
# ).start()
# !pip -q install pyngrok
# from pyngrok import ngrok
# from google.colab import userdata
# ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
# print("UI:", ngrok.connect(7860).public_url)
print("UI disabled by default — uncomment above if needed.")

### Docs
- Full guide: `documentation/COLAB.md`
- Fast env template: `configuration/colab_fast.env.example`
- Project flow: `documentation/PROJECT_FLOW.md`